This notebook includes the code to reproduce the results reported on the appendix regarding the quantization tables.

In [ ]:
import os
from PIL import Image
import numpy as np
from collections import defaultdict, Counter

In [ ]:
def get_qtables(path):
    return Image.open(path).quantization


def tables_to_tuple(qtables):
    return tuple(
        (k, tuple(qtables[k]))
        for k in sorted(qtables.keys())
    )


def process_dataset(root):
    sensor_tables = {}
    sensor_all_occurrences = {}

    for sensor in os.listdir(root):
        sensor_path = os.path.join(root, sensor)
        if not os.path.isdir(sensor_path):
            continue

        tables_counter = Counter()
        all_seen = defaultdict(list)

        print(f"\nProcessing sensor: {sensor}")

        for r, _, files in os.walk(sensor_path):
            for f in files:
                if f.lower().endswith(".jpg"):
                    path = os.path.join(r, f)

                    try:
                        qtables = get_qtables(path)
                        key = tables_to_tuple(qtables)

                        tables_counter[key] += 1
                        all_seen[key].append(path)

                    except Exception as e:
                        print(f"Error reading {path}: {e}")

        # summary warning
        if len(tables_counter) > 1:
            print("WARNING: Multiple quantization tables found in this sensor")

        sensor_tables[sensor] = tables_counter
        sensor_all_occurrences[sensor] = all_seen

    return sensor_tables, sensor_all_occurrences


def group_by_model(sensor_tables):
    model_groups = defaultdict(dict)

    for sensor, tables in sensor_tables.items():
        model = sensor[0]  # first letter
        model_groups[model][sensor] = tables

    return model_groups


def print_sensor_summary(sensor_tables, all_occurrences):
    for sensor, tables in sensor_tables.items():
        print(f"\n==============================")
        print(f"SENSOR: {sensor}")
        print(f"==============================")

        print(f"Number of distinct quantization tables: {len(tables)}")

        if len(tables) > 1:
            print("WARNING: Multiple quantization tables found in this sensor")

        for i, (table_tuple, count) in enumerate(tables.items()):
            print(f"\n--- Table #{i+1} (appears {count} times) ---")

            paths = all_occurrences[sensor][table_tuple]

            print(f"Appears in {len(paths)} files:")
            for p in paths[:10]:  # limit to first 10
                print(f"  - {p}")
            if len(paths) > 10:
                print(f"  ... and {len(paths) - 10} more")

            qtables = dict(table_tuple)

            print("\nLuminance:")
            print(np.array(qtables[0]).reshape(8, 8))

            if 1 in qtables:
                print("\nChrominance:")
                print(np.array(qtables[1]).reshape(8, 8))


def print_model_summary(model_groups):
    for model, sensors in model_groups.items():
        print(f"\n########################################")
        print(f"MODEL: {model}")
        print(f"########################################")

        # merge all tables
        all_tables = Counter()
        for tables in sensors.values():
            all_tables.update(tables)

        print(f"Total distinct tables in model: {len(all_tables)}")

        if len(all_tables) > 1:
            print("WARNING: Sensors of this model use different quantization tables")

        # compare sensors
        sensor_sets = {
            s: set(t.keys()) for s, t in sensors.items()
        }

        sensors_list = list(sensor_sets.keys())
        base_sensor = sensors_list[0]
        base_set = sensor_sets[base_sensor]

        for sensor, table_set in sensor_sets.items():
            if table_set != base_set:
                print(f"Difference between sensors: {base_sensor} vs {sensor}")

In [ ]:
sensor_tables, first_occurrence = process_dataset(PATH)

print_sensor_summary(sensor_tables, first_occurrence)

model_groups = group_by_model(sensor_tables)

print_model_summary(model_groups)